# Validation

In [1]:
import pandas as pd
import re

In [2]:
# Paths
PATH = '/home/tatiane/repositorios/brazil-health-data-project/data_pipeline'
raw_path = f"{PATH}/data/raw/ibge_pnad_continua/tabela5436.csv"

In [3]:
# Reading
raw = pd.read_csv(raw_path, sep=",", encoding="latin1")

### Validation

- First validation

In [4]:
print(len(raw))
raw.head()

27


,ï»¿CÃ³d.,Unidade da FederaÃ§Ã£o,1Âº trimestre 2018,2Âº trimestre 2018,3Âº trimestre 2018,4Âº trimestre 2018,1Âº trimestre 2019,2Âº trimestre 2019,3Âº trimestre 2019,4Âº trimestre 2019,1Âº trimestre 2020,2Âº trimestre 2022,3Âº trimestre 2022,4Âº trimestre 2022
0,11,RondÃ´nia,2699,2663,2787,2719,2601,2622,2739,2709,2662,2658,2755,2900
1,12,Acre,2352,2401,2466,2476,2578,2544,2510,2595,2461,2641,2587,2538
2,13,Amazonas,2442,2439,2321,2277,2335,2288,2352,2294,2320,2258,2333,2409
3,14,Roraima,3185,3059,3011,2976,2935,3136,3146,2926,2995,2708,2602,2936
4,15,ParÃ¡,2109,2063,2096,2052,2082,2027,2093,2156,2113,2125,2217,2243


In [5]:
# Primeiras transformações realizadas na base
raw_change = raw.rename(columns={
    raw.columns[0]: "cod_uf",
    raw.columns[1]: "uf"
})
raw_change.columns = raw_change.columns.str.encode('latin1').str.decode('utf-8')
raw_change['uf'] = raw_change['uf'].str.encode('latin1').str.decode('utf-8')

In [6]:
print(len(raw_change))
raw_change.head()

27


,cod_uf,uf,1º trimestre 2018,2º trimestre 2018,3º trimestre 2018,4º trimestre 2018,1º trimestre 2019,2º trimestre 2019,3º trimestre 2019,4º trimestre 2019,1º trimestre 2020,2º trimestre 2022,3º trimestre 2022,4º trimestre 2022
0,11,Rondônia,2699,2663,2787,2719,2601,2622,2739,2709,2662,2658,2755,2900
1,12,Acre,2352,2401,2466,2476,2578,2544,2510,2595,2461,2641,2587,2538
2,13,Amazonas,2442,2439,2321,2277,2335,2288,2352,2294,2320,2258,2333,2409
3,14,Roraima,3185,3059,3011,2976,2935,3136,3146,2926,2995,2708,2602,2936
4,15,Pará,2109,2063,2096,2052,2082,2027,2093,2156,2113,2125,2217,2243


In [7]:
diferencas = raw.reset_index(drop=True).equals(raw_change.reset_index(drop=True))
print(diferencas)

False


- Second Validation 

In [8]:
anos = sorted(set([
    re.search(r'\d{4}', col).group()
    for col in raw_change.columns if re.search(r'\d{4}', col)
]))

In [9]:
for ano in anos:
    cols_ano = [col for col in raw_change.columns if ano in col]
    raw_change[ano] = raw_change[cols_ano].sum(axis=1)

In [10]:
raw_change.head()

,cod_uf,uf,1º trimestre 2018,2º trimestre 2018,3º trimestre 2018,4º trimestre 2018,1º trimestre 2019,2º trimestre 2019,3º trimestre 2019,4º trimestre 2019,1º trimestre 2020,2º trimestre 2022,3º trimestre 2022,4º trimestre 2022,2018,2019,2020,2022
0,11,Rondônia,2699,2663,2787,2719,2601,2622,2739,2709,2662,2658,2755,2900,10868,10671,2662,8313
1,12,Acre,2352,2401,2466,2476,2578,2544,2510,2595,2461,2641,2587,2538,9695,10227,2461,7766
2,13,Amazonas,2442,2439,2321,2277,2335,2288,2352,2294,2320,2258,2333,2409,9479,9269,2320,7000
3,14,Roraima,3185,3059,3011,2976,2935,3136,3146,2926,2995,2708,2602,2936,12231,12143,2995,8246
4,15,Pará,2109,2063,2096,2052,2082,2027,2093,2156,2113,2125,2217,2243,8320,8358,2113,6585


In [11]:
anos = [2018, 2019]

for ano in anos:

    colunas_trimestre = [
        f"1º trimestre {ano}",
        f"2º trimestre {ano}",
        f"3º trimestre {ano}",
        f"4º trimestre {ano}"
    ]

    raw_change[f"check_{ano}"] = raw_change[colunas_trimestre].sum(axis=1)

    erros = raw_change[
        raw_change[str(ano)] != raw_change[f"check_{ano}"]
    ]

    print(f"Ano {ano} -> erros: {len(erros)}")

Ano 2018 -> erros: 0
Ano 2019 -> erros: 0


In [12]:
raw_change["check_2022"] = (
    raw_change["2º trimestre 2022"] +
    raw_change["3º trimestre 2022"] +
    raw_change["4º trimestre 2022"]
)

raw_change[raw_change["2022"] != raw_change["check_2022"]]

,cod_uf,uf,1º trimestre 2018,2º trimestre 2018,3º trimestre 2018,4º trimestre 2018,1º trimestre 2019,2º trimestre 2019,3º trimestre 2019,4º trimestre 2019,...,2º trimestre 2022,3º trimestre 2022,4º trimestre 2022,2018,2019,2020,2022,check_2018,check_2019,check_2022


- Validar valores ausentes

In [13]:
anos = [2018, 2019, 2020, 2022]

for ano in anos:

    colunas_trimestre = [
        col
        for col in raw_change.columns
        if str(ano) in col and "trimestre" in col
    ]

    print(f"\n===== ANO {ano} =====")

    # quantidade de nulos por trimestre
    for coluna in colunas_trimestre:

        qtd_nulos = raw_change[coluna].isnull().sum()

        print(f"{coluna}: {qtd_nulos} nulos")

    # valida coluna anual
    if str(ano) in raw_change.columns:

        nulos_ano = raw_change[str(ano)].isnull().sum()

        print(f"Coluna {ano}: {nulos_ano} nulos")


===== ANO 2018 =====
1º trimestre 2018: 0 nulos
2º trimestre 2018: 0 nulos
3º trimestre 2018: 0 nulos
4º trimestre 2018: 0 nulos
Coluna 2018: 0 nulos

===== ANO 2019 =====
1º trimestre 2019: 0 nulos
2º trimestre 2019: 0 nulos
3º trimestre 2019: 0 nulos
4º trimestre 2019: 0 nulos
Coluna 2019: 0 nulos

===== ANO 2020 =====
1º trimestre 2020: 0 nulos
Coluna 2020: 0 nulos

===== ANO 2022 =====
2º trimestre 2022: 0 nulos
3º trimestre 2022: 0 nulos
4º trimestre 2022: 0 nulos
Coluna 2022: 0 nulos


- Validação da soma anual

In [14]:
anos = [2018, 2019, 2020, 2022]

for ano in anos:

    colunas_trimestre = [
        col
        for col in raw_change.columns
        if str(ano) in col and "trimestre" in col
    ]

    raw_change[f"check_{ano}"] = (
        raw_change[colunas_trimestre]
        .sum(axis=1)
    )

    erros = raw_change[
        raw_change[str(ano)] !=
        raw_change[f"check_{ano}"]
    ]

    print(f"\nAno {ano}")
    print(f"Quantidade de erros: {len(erros)}")

    if len(erros) > 0:
        display(
            erros[
                ["uf", str(ano), f"check_{ano}"]
            ]
        )


Ano 2018
Quantidade de erros: 0

Ano 2019
Quantidade de erros: 0

Ano 2020
Quantidade de erros: 0

Ano 2022
Quantidade de erros: 0
